In [29]:
from transformers import AutoModelForCausalLM, AutoTokenizer, QuantoConfig, VitsModel, TextStreamer
import time
import psutil
import torch
import whisper
import sounddevice as sd
import numpy as np
from IPython.display import Audio
from queue import Queue
import threading
import numpy as np
from openai import OpenAI


In [2]:
# --- LLAMA ---

model_id = "meta-llama/Llama-3.2-1B-Instruct"
tokenizer = AutoTokenizer.from_pretrained(model_id)
quantization_config = QuantoConfig(weights="int8")

model = AutoModelForCausalLM.from_pretrained(model_id,
    device_map="cpu",
    quantization_config=quantization_config)

In [30]:
client = OpenAI(
    base_url="http://127.0.0.1:8080/v1",
    api_key = "sk-no-key-required"
)

In [31]:
# --- STT ---

whisper_model = whisper.load_model("small")

def perintah():
    duration = 5
    sample_rate = 16000  

    print("Mendengarkan......")
    audio_data = sd.rec(int(duration * sample_rate), samplerate=sample_rate, channels=1, dtype='float32')
    sd.wait()  
    print("Diterima.....")

    audio_data = np.squeeze(audio_data)  
    dengar = whisper_model.transcribe(audio_data, fp16=False, language="id")

    return dengar["text"], audio_data

c:\Users\Alysha\Documents\kata-ondevice\myenv\lib\site-packages\whisper\__init__.py:150: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  checkpoint = torch.load(fp, map_locati

In [32]:
# --- TTS ---

mms = VitsModel.from_pretrained("facebook/mms-tts-ind")
mms_tokenizer = AutoTokenizer.from_pretrained("facebook/mms-tts-ind")

# def ngomong(text):
#     inputs = mms_token(text, return_tensors="pt")

#     with torch.no_grad():
#         output = mms(**inputs).waveform
#         return Audio(output.squeeze().cpu().numpy(), rate=16000) # Can change rate to make it faster/slower

def ngomong(text: str):
    """
    Converts text to speech using the MMS TTS model (facebook/mms-tts-ind) and plays the audio.
    Args:
        text (str): The input text to be synthesized into speech.
    """
    inputs = mms_tokenizer(text, return_tensors="pt")
    
    with torch.no_grad():
        output = mms(**inputs).waveform
    
    audio_array = output.squeeze().cpu().numpy()
    
    sample_rate = 16000  # You can adjust this based on the model's specifications if necessary
    
    sd.play(audio_array, sample_rate)
    sd.wait() 


In [33]:
def get_llm_response(text: str) -> str:
    """
    Generates a response to the given text using the Llama-2 language model.
    Args:
        text (str): The input text to be processed.
    Returns:
        str: The generated response.
    """
    # Prepare the prompt
    prompt = f"User: {text}\nAssistant: Tolong jawab singkat kurang dari 20 kata."

    # Tokenize input and generate response
    inputs = tokenizer(prompt, return_tensors="pt")
    output = model.generate(
        inputs.input_ids,
        max_length=150,
        temperature=0.7,
        top_p=0.9,
        do_sample=True
    )
    
    response = tokenizer.decode(output[0], skip_special_tokens=True)
    
    # Extract assistant's response
    response = response.split("Assistant:")[-1].strip()
    return response

In [40]:
def get_llm_response(text: str) -> str:
     completion = client.chat.completions.create(
        model="LLaMA_CPP",
        messages=[
            {"role": "system", "content": "Tolong jawab dengan singkat"},
            {"role": "user", "content": text}
        ]
    )
     
    #  print("LLM result: {0}".format(completion.choices[0].message))
     return completion.choices[0].message.content
     

In [35]:
def main_loop():
    """Main loop for the app"""
    try:
        while True:
            input("Press Enter to start recording.")

            # Process the recorded audio and transcription
            text, audio_data = perintah()

            if audio_data.size > 0:
                print(f"You: {text}")
                print("Generating response...")
                
                response = get_llm_response(text)
                print(f"Assistant: {response}")
                
                ngomong(response) 
            else:
                print("No audio recorded. Please ensure your microphone is working.")
                
    except KeyboardInterrupt:
        print("\nExiting...")

In [41]:
if __name__ == "__main__":
    main_loop()

Mendengarkan......
Diterima.....
You:  Halo, apakah bar?
Generating response...
Assistant: Tentu, aku bisa membantu kamu. Apa yang ingin kamu cari di bar?<|eot_id|>
Mendengarkan......
Diterima.....
You:  gimana cara memasak nasi goreng?
Generating response...
Assistant: Cara memasak nasi goreng:

1. Panaskan minyak dalam wajan, tambahkan bawang putih, lada, dan garam.
2. Masukkan nasi yang sudah direbus, aduk rata.
3. Tambahkan telur, aduk rata.
4. Masukkan sayuran (seperti wortel, buncis, atau tomat), aduk rata.
5. Tambahkan kecap manis, saus tiram, dan gula pasir.
6. Aduk rata dan biarkan mendidih selama 2-3 menit.
7. Matikan api dan sajikan nasi goreng dengan sate, ayam, atau telur goreng.<|eot_id|>
Mendengarkan......

Exiting...
